<a href="https://colab.research.google.com/github/N3ktye/Ml-for-elogistic/blob/main/Hackethon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests

In [ ]:
import requests

API_KEY = "dde8af6ead84f0191694225015f965e6"

city = "Bhubaneswar"

url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

response = requests.get(url)

print(response.json())

{'coord': {'lon': 85.8333, 'lat': 20.2333}, 'weather': [{'id': 721, 'main': 'Haze', 'description': 'haze', 'icon': '50d'}], 'base': 'stations', 'main': {'temp': 36.12, 'feels_like': 35.42, 'temp_min': 36.12, 'temp_max': 36.12, 'pressure': 1007, 'humidity': 26, 'sea_level': 1007, 'grnd_level': 1003}, 'visibility': 3500, 'wind': {'speed': 3.09, 'deg': 300}, 'clouds': {'all': 40}, 'dt': 1773299111, 'sys': {'type': 1, 'id': 9113, 'country': 'IN', 'sunrise': 1773275290, 'sunset': 1773318305}, 'timezone': 19800, 'id': 1275817, 'name': 'Bhubaneswar', 'cod': 200}


In [ ]:
import requests

API_KEY = "dde8af6ead84f0191694225015f965e6"

def get_weather(city):

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

    response = requests.get(url)
    data = response.json()

    if str(data.get("cod")) != "200":
        print("API Error:", data.get("message"))
        return None, None

    weather = data["weather"][0]["main"]
    temp = data["main"]["temp"]

    return weather, temp


weather, temp = get_weather("Bhubaneswar")

print("Weather:", weather)
print("Temperature:", temp, "°C")

Weather: Haze
Temperature: 36.12 °C


In [ ]:
!pip install openrouteservice

In [ ]:
import openrouteservice

API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjE2ODlhYzhiMTQ3MzQyYWI5ZTc2NTQ3ODU0ZjUyYTVmIiwiaCI6Im11cm11cjY0In0="

client = openrouteservice.Client(key=API_KEY)

# Bhubaneswar → Kolkata coordinates
coords = ((85.8245, 20.2961), (88.3639, 22.5726))

route = client.directions(coords)

distance = route['routes'][0]['summary']['distance'] / 1000
duration = route['routes'][0]['summary']['duration'] / 3600

print("Distance:", round(distance,2), "km")
print("Travel Time:", round(duration,2), "hours")

Distance: 440.03 km
Travel Time: 5.5 hours


In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
import random

weather_conditions = ["Clear", "Clouds", "Rain", "Storm", "Haze"]
shipment_types = ["Normal", "Fragile", "Express"]

data = []

for i in range(200):

    distance = random.randint(100,1000)
    travel_time = random.uniform(2,15)
    weather = random.choice(weather_conditions)
    shipment = random.choice(shipment_types)
    past_delay = random.randint(0,1)

    # simple delay rule
    delay = 1 if weather in ["Rain","Storm"] or travel_time > 8 else 0

    data.append([distance,travel_time,weather,shipment,past_delay,delay])

df = pd.DataFrame(data,columns=[
    "distance",
    "travel_time",
    "weather",
    "shipment_type",
    "past_delay",
    "delay"
])

print(df.head())

   distance  travel_time weather shipment_type  past_delay  delay
0       703     3.734803  Clouds       Fragile           0      0
1      1000     5.393283    Rain       Fragile           0      1
2       122    10.746304   Clear       Express           0      1
3       765     9.260282   Clear       Fragile           1      1
4       236     3.961686    Rain       Express           1      1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["weather"] = le.fit_transform(df["weather"])
df["shipment_type"] = le.fit_transform(df["shipment_type"])

X = df.drop("delay",axis=1)
y = df["delay"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train,y_train)

accuracy = model.score(X_test,y_test)

print("Model Accuracy:",accuracy)

Model Accuracy: 0.975


In [ ]:
def predict_delay(distance, travel_time, weather, shipment_type, past_delay):

    weather = le.transform([weather])[0]
    shipment_type = le.transform([shipment_type])[0]

    input_data = [[distance, travel_time, weather, shipment_type, past_delay]]

    prediction = model.predict(input_data)
    probability = model.predict_proba(input_data)

    return prediction[0], probability[0][1]

In [ ]:
from sklearn.preprocessing import LabelEncoder

weather_encoder = LabelEncoder()
shipment_encoder = LabelEncoder()

df["weather"] = weather_encoder.fit_transform(df["weather"])
df["shipment_type"] = shipment_encoder.fit_transform(df["shipment_type"])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop("delay", axis=1)
y = df["delay"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [ ]:
def predict_delay(distance, travel_time, weather, shipment_type, past_delay):

    weather_num = weather_encoder.transform([weather])[0]
    shipment_num = shipment_encoder.transform([shipment_type])[0]

    input_data_df = pd.DataFrame({
        "distance":[distance],
        "travel_time":[travel_time],
        "weather":[weather_num],
        "shipment_type":[shipment_num],
        "past_delay":[past_delay]
    })

    prediction = model.predict(input_data_df)[0]
    probability = model.predict_proba(input_data_df)[0][1]

    return prediction, probability

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Re-fit the encoders to ensure they are correctly mapped to string labels
# weather_conditions and shipment_types are assumed to be globally available
weather_encoder = LabelEncoder()
shipment_encoder = LabelEncoder()

# Fit on the original string categories
weather_encoder.fit(weather_conditions)
shipment_encoder.fit(shipment_types)

distance = 440
travel_time = 5.5
weather = "Haze"
shipment_type = "Express"
past_delay = 0

result, risk = predict_delay(distance, travel_time, weather, shipment_type, past_delay)

print("Delay Prediction:", result)
print("Delay Risk:", round(risk*100,2), "%")

Delay Prediction: 0
Delay Risk: 7.0 %
